# 🧬 Dark Manifold V26.1 — FAIR TEST with Complex Kinetics

## Critical Change from V26

V26 achieved r=0.999 BUT the test was rigged: the ground truth ODE used the 
same functional form (Michaelis-Menten) as the model. This is parameter 
estimation, not dynamics prediction.

### V26.1 Fair Test Design

| Component | V26 Ground Truth | V26.1 Ground Truth |
|-----------|-----------------|-------------------|
| Kinetics | Simple MM | **Hill + Product Inhibition** |
| Cooperativity | None | **n = 1.0-2.5** |
| Product inhibition | None | **Ki varies per rxn** |
| Substrate inhibition | None | **30% of reactions** |
| Allosteric regulation | Simple | **Energy charge dependent** |
| NADH coupling | None | **TCA inhibition** |

### The Test

The V26.1 model STILL uses simple MM + neural residual.
But the ground truth uses complex kinetics.

**Question:** Can a simple model approximate complex dynamics?

If yes → Physics-first approach is robust
If no → Need better architecture

This is how science should work.


In [ ]:
#@title 1️⃣ Setup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#@title 2️⃣ Biochemistry with Pathway Structure

class StructuredBiochemistry:
    """Biochemistry with explicit pathway structure.
    
    Unlike random matrices, this has REAL metabolic organization:
    - Glycolysis (linear pathway)
    - TCA cycle (circular pathway)  
    - Biosynthesis (branching pathways)
    - Energy coupling (ATP/ADP balance)
    """
    
    def __init__(self):
        # Metabolite definitions with biological meaning
        self.metabolites = [
            # Energy carriers (indices 0-3)
            'ATP', 'ADP', 'NAD+', 'NADH',
            # Glycolysis (indices 4-13)
            'Glucose', 'G6P', 'F6P', 'FBP', 'DHAP', 'G3P', 
            'BPG', 'PG3', 'PEP', 'Pyruvate',
            # TCA cycle (indices 14-21)
            'AcCoA', 'Citrate', 'Isocitrate', 'aKG', 
            'SuccCoA', 'Succinate', 'Fumarate', 'Malate',
            # Biosynthesis precursors (indices 22-29)
            'Ribose5P', 'PRPP', 'IMP', 'AMP', 'GMP',
            'Serine', 'Glycine', 'Alanine'
        ]
        
        self.n_metabolites = len(self.metabolites)
        self.met_idx = {m: i for i, m in enumerate(self.metabolites)}
        
        # Define reactions with explicit stoichiometry
        self.reactions = []
        self._build_glycolysis()
        self._build_tca_cycle()
        self._build_biosynthesis()
        
        self.n_reactions = len(self.reactions)
        
        # Build stoichiometry matrix
        self.S = np.zeros((self.n_metabolites, self.n_reactions))
        for j, rxn in enumerate(self.reactions):
            for met, coef in rxn['stoich'].items():
                self.S[self.met_idx[met], j] = coef
        
        # Define genes (1-2 genes per reaction)
        self.n_genes = 25
        self.G = np.zeros((self.n_genes, self.n_reactions))
        np.random.seed(42)
        for j in range(self.n_reactions):
            n_genes = np.random.choice([1, 1, 2])
            genes = np.random.choice(self.n_genes, n_genes, replace=False)
            self.G[genes, j] = 1
        
        # Identify pathway membership
        self.glycolysis_rxns = list(range(10))
        self.tca_rxns = list(range(10, 18))
        self.biosyn_rxns = list(range(18, self.n_reactions))
        
        print(f"📊 Structured Biochemistry:")
        print(f"   {self.n_metabolites} metabolites, {self.n_reactions} reactions, {self.n_genes} genes")
        print(f"   Glycolysis: {len(self.glycolysis_rxns)} rxns")
        print(f"   TCA cycle: {len(self.tca_rxns)} rxns")
        print(f"   Biosynthesis: {len(self.biosyn_rxns)} rxns")
    
    def _add_reaction(self, name, stoich, reversible=False):
        self.reactions.append({
            'name': name,
            'stoich': stoich,
            'reversible': reversible
        })
    
    def _build_glycolysis(self):
        """Glycolysis: Glucose → 2 Pyruvate + 2 ATP + 2 NADH"""
        # Hexokinase: Glucose + ATP → G6P + ADP
        self._add_reaction('HK', {'Glucose': -1, 'ATP': -1, 'G6P': 1, 'ADP': 1})
        # PGI: G6P ⇌ F6P
        self._add_reaction('PGI', {'G6P': -1, 'F6P': 1}, reversible=True)
        # PFK: F6P + ATP → FBP + ADP
        self._add_reaction('PFK', {'F6P': -1, 'ATP': -1, 'FBP': 1, 'ADP': 1})
        # Aldolase: FBP ⇌ DHAP + G3P
        self._add_reaction('ALDO', {'FBP': -1, 'DHAP': 1, 'G3P': 1}, reversible=True)
        # TPI: DHAP ⇌ G3P
        self._add_reaction('TPI', {'DHAP': -1, 'G3P': 1}, reversible=True)
        # GAPDH: G3P + NAD+ → BPG + NADH
        self._add_reaction('GAPDH', {'G3P': -1, 'NAD+': -1, 'BPG': 1, 'NADH': 1})
        # PGK: BPG + ADP → PG3 + ATP
        self._add_reaction('PGK', {'BPG': -1, 'ADP': -1, 'PG3': 1, 'ATP': 1})
        # PGM: PG3 ⇌ PEP
        self._add_reaction('PGM', {'PG3': -1, 'PEP': 1}, reversible=True)
        # Enolase implicit in PGM
        # PK: PEP + ADP → Pyruvate + ATP
        self._add_reaction('PK', {'PEP': -1, 'ADP': -1, 'Pyruvate': 1, 'ATP': 1})
        # PDH: Pyruvate + NAD+ → AcCoA + NADH
        self._add_reaction('PDH', {'Pyruvate': -1, 'NAD+': -1, 'AcCoA': 1, 'NADH': 1})
    
    def _build_tca_cycle(self):
        """TCA Cycle: AcCoA → CO2 + ATP + NADH"""
        # Citrate synthase: AcCoA + OAA → Citrate
        # (OAA comes from Malate, so cycle is closed)
        self._add_reaction('CS', {'AcCoA': -1, 'Malate': -1, 'Citrate': 1})
        # Aconitase: Citrate ⇌ Isocitrate
        self._add_reaction('ACO', {'Citrate': -1, 'Isocitrate': 1}, reversible=True)
        # IDH: Isocitrate + NAD+ → aKG + NADH
        self._add_reaction('IDH', {'Isocitrate': -1, 'NAD+': -1, 'aKG': 1, 'NADH': 1})
        # aKGDH: aKG + NAD+ → SuccCoA + NADH
        self._add_reaction('AKGDH', {'aKG': -1, 'NAD+': -1, 'SuccCoA': 1, 'NADH': 1})
        # SCS: SuccCoA + ADP → Succinate + ATP
        self._add_reaction('SCS', {'SuccCoA': -1, 'ADP': -1, 'Succinate': 1, 'ATP': 1})
        # SDH: Succinate → Fumarate
        self._add_reaction('SDH', {'Succinate': -1, 'Fumarate': 1})
        # FUM: Fumarate ⇌ Malate
        self._add_reaction('FUM', {'Fumarate': -1, 'Malate': 1}, reversible=True)
        # MDH: Malate + NAD+ → OAA + NADH (OAA feeds back to CS)
        self._add_reaction('MDH', {'Malate': -1, 'NAD+': -1, 'NADH': 1})
    
    def _build_biosynthesis(self):
        """Biosynthesis pathways branching from central metabolism."""
        # Pentose phosphate: G6P → Ribose5P
        self._add_reaction('G6PDH', {'G6P': -1, 'Ribose5P': 1})
        # PRPP synthesis: Ribose5P + ATP → PRPP + ADP
        self._add_reaction('PRPS', {'Ribose5P': -1, 'ATP': -1, 'PRPP': 1, 'ADP': 1})
        # Purine synthesis: PRPP → IMP
        self._add_reaction('PPAT', {'PRPP': -1, 'ATP': -1, 'IMP': 1, 'ADP': 1})
        # IMP → AMP
        self._add_reaction('ADSS', {'IMP': -1, 'ATP': -1, 'AMP': 1, 'ADP': 1})
        # IMP → GMP
        self._add_reaction('GMPS', {'IMP': -1, 'ATP': -1, 'GMP': 1, 'ADP': 1})
        # Serine synthesis from PG3
        self._add_reaction('PHGDH', {'PG3': -1, 'NAD+': -1, 'Serine': 1, 'NADH': 1})
        # Glycine from Serine
        self._add_reaction('SHMT', {'Serine': -1, 'Glycine': 1})
        # Alanine from Pyruvate
        self._add_reaction('ALT', {'Pyruvate': -1, 'Alanine': 1})

biochem = StructuredBiochemistry()

In [ ]:
#@title 3️⃣ Layer 1 & 2: Classical Kinetics (Analytical + Few Parameters)

class ClassicalKinetics(nn.Module):
    """Classical enzyme kinetics — the PHYSICS layer.
    
    This is NOT a neural network. It implements:
    - Michaelis-Menten kinetics
    - Gene expression modulation
    - ATP/ADP ratio effects
    
    Only ~200 learnable parameters (Vmax, Km per reaction).
    """
    
    def __init__(self, S, G, met_idx):
        super().__init__()
        
        n_met, n_rxn = S.shape
        n_gene = G.shape[0]
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        self.n_met = n_met
        self.n_rxn = n_rxn
        self.n_gene = n_gene
        self.met_idx = met_idx
        
        # Kinetic parameters (learnable but constrained)
        # Vmax: maximum velocity (positive)
        self.log_vmax = nn.Parameter(torch.zeros(n_rxn))
        # Km: Michaelis constant (positive)
        self.log_km = nn.Parameter(torch.zeros(n_rxn))
        
        # Precompute substrate indices for each reaction
        self.substrate_idx = []
        for j in range(n_rxn):
            subs = torch.where(self.S[:, j] < 0)[0]
            self.substrate_idx.append(subs)
    
    def forward(self, M, G_expr):
        """
        Compute flux using Michaelis-Menten kinetics.
        
        Args:
            M: Metabolite concentrations (batch, n_met)
            G_expr: Gene expression levels (batch, n_gene)
        
        Returns:
            v: Flux vector (batch, n_rxn)
        """
        batch = M.shape[0]
        
        # Get kinetic parameters (constrained positive)
        vmax = torch.exp(self.log_vmax)  # (n_rxn,)
        km = torch.exp(self.log_km) + 0.1  # (n_rxn,), avoid division by zero
        
        # Compute flux for each reaction
        v = torch.zeros(batch, self.n_rxn, device=M.device)
        
        for j in range(self.n_rxn):
            # Get substrate concentration (use first substrate for simplicity)
            subs = self.substrate_idx[j]
            if len(subs) > 0:
                S_conc = M[:, subs[0]]  # (batch,)
            else:
                S_conc = torch.ones(batch, device=M.device)
            
            # Michaelis-Menten: v = Vmax * S / (Km + S)
            v[:, j] = vmax[j] * S_conc / (km[j] + S_conc)
        
        # Gene expression modulation: v *= enzyme_level
        # Enzyme level = sigmoid(weighted sum of gene expressions)
        enzyme_level = torch.sigmoid(G_expr @ self.G)  # (batch, n_rxn)
        v = v * enzyme_level
        
        # Energy charge regulation: ATP/ADP ratio affects flux
        atp_idx = self.met_idx.get('ATP', 0)
        adp_idx = self.met_idx.get('ADP', 1)
        energy_charge = M[:, atp_idx] / (M[:, atp_idx] + M[:, adp_idx] + 0.01)
        
        # High energy charge inhibits catabolic reactions (glycolysis)
        # Low energy charge inhibits anabolic reactions (biosynthesis)
        # For simplicity, just scale all fluxes by a factor
        v = v * (0.5 + 0.5 * energy_charge.unsqueeze(1))
        
        return v

# Test
classical = ClassicalKinetics(biochem.S, biochem.G, biochem.met_idx)
n_params = sum(p.numel() for p in classical.parameters())
print(f"✅ Classical kinetics: {n_params} parameters")

In [ ]:
#@title 4️⃣ Layer 3: Neural Residual (SMALL Network)

class NeuralResidual(nn.Module):
    """Small neural network that learns CORRECTIONS to classical kinetics.
    
    This captures:
    - Enzyme-enzyme interactions (not in classical model)
    - Allosteric regulation (feedback/feedforward)
    - Complex regulatory circuits
    
    Key design choices:
    - SMALL: ~10K parameters (not 8M like V24)
    - RESIDUAL: Adds to classical, doesn't replace
    - BOUNDED: Output scaled to be a small correction
    """
    
    def __init__(self, n_met, n_gene, n_rxn, hidden_dim=64):
        super().__init__()
        
        input_dim = n_met + n_gene
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_rxn),
            nn.Tanh()  # Bounded output [-1, 1]
        )
        
        # Learnable scale for residual (start small)
        self.residual_scale = nn.Parameter(torch.tensor(0.1))
    
    def forward(self, M, G_expr):
        """
        Compute residual correction to flux.
        
        Returns:
            delta_v: Small correction to add to classical flux
        """
        x = torch.cat([M, G_expr], dim=-1)
        delta_v = self.net(x)
        
        # Scale to be a small correction (10-20% of classical)
        delta_v = delta_v * torch.sigmoid(self.residual_scale)
        
        return delta_v

# Test
residual = NeuralResidual(biochem.n_metabolites, biochem.n_genes, biochem.n_reactions)
n_params = sum(p.numel() for p in residual.parameters())
print(f"✅ Neural residual: {n_params} parameters")

In [ ]:
#@title 5️⃣ V26 Full Model: Physics + Residual

class DarkManifoldV26(nn.Module):
    """V26: Physics-First with Neural Residual
    
    Architecture:
        v_total = v_classical + v_residual
        dM/dt = S @ v_total
    
    The classical layer provides 90% of the prediction.
    The neural residual provides the remaining 10%.
    
    This is similar to:
    - ResNet: y = f(x) + x
    - PINN: Learn residual of known PDE
    - Hybrid models: Physics + ML
    """
    
    def __init__(self, S, G, met_idx, hidden_dim=64, dt=0.1):
        super().__init__()
        
        n_met, n_rxn = S.shape
        n_gene = G.shape[0]
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.n_met = n_met
        self.n_gene = n_gene
        self.n_rxn = n_rxn
        
        # Layer 1+2: Classical kinetics
        self.classical = ClassicalKinetics(S, G, met_idx)
        
        # Layer 3: Neural residual
        self.residual = NeuralResidual(n_met, n_gene, n_rxn, hidden_dim)
        
        # Gene dynamics (simple mean-reverting)
        self.gene_decay = nn.Parameter(torch.tensor(0.01))
        
        # Time step
        self.dt = dt
    
    def forward(self, M, G_expr, return_components=False):
        """
        One step of dynamics.
        
        Args:
            M: Metabolite concentrations (batch, n_met)
            G_expr: Gene expression (batch, n_gene)
            return_components: If True, return classical and residual separately
        
        Returns:
            new_M: Updated metabolites
            new_G: Updated genes
            (optionally) v_classical, v_residual
        """
        # Compute fluxes
        v_classical = self.classical(M, G_expr)
        v_residual = self.residual(M, G_expr)
        
        # Total flux
        v_total = v_classical + v_residual
        
        # Stoichiometric dynamics: dM/dt = S @ v
        dM = v_total @ self.S.T
        
        # Update metabolites
        new_M = M + self.dt * dM
        new_M = torch.clamp(new_M, 0.001, 100.0)  # Keep positive
        
        # Gene dynamics: slow mean-reversion to baseline
        dG = -self.gene_decay * (G_expr - 1.0)
        new_G = G_expr + self.dt * dG
        new_G = torch.clamp(new_G, 0.01, 10.0)
        
        if return_components:
            return new_M, new_G, v_classical, v_residual
        return new_M, new_G
    
    def trajectory(self, M0, G0, n_steps):
        """Generate full trajectory."""
        M_traj = [M0]
        G_traj = [G0]
        
        M, G = M0, G0
        for _ in range(n_steps - 1):
            M, G = self.forward(M, G)
            M_traj.append(M)
            G_traj.append(G)
        
        return torch.stack(M_traj, dim=1), torch.stack(G_traj, dim=1)

# Test
model = DarkManifoldV26(biochem.S, biochem.G, biochem.met_idx)
n_params = sum(p.numel() for p in model.parameters())
print(f"✅ V26 Total: {n_params:,} parameters")
print(f"   Classical: {sum(p.numel() for p in model.classical.parameters())}")
print(f"   Residual: {sum(p.numel() for p in model.residual.parameters())}")
print(f"\n   Compare to V24: ~8,000,000 parameters")
print(f"   V26 is {8_000_000 / n_params:.0f}x smaller!")

In [ ]:
#@title 6️⃣ V26.1: FAIR TEST - Complex Ground Truth Kinetics

class ComplexGroundTruthODE:
    """V26.1: Generate ground truth with COMPLEX kinetics.
    
    FAIR TEST: The ground truth uses kinetics that are DIFFERENT from
    the model's simple Michaelis-Menten. This tests whether the model
    can generalize to dynamics it wasn't designed for.
    
    Complex kinetics include:
    1. Hill kinetics (cooperativity)
    2. Product inhibition
    3. Allosteric regulation (ATP/AMP energy charge)
    4. Substrate inhibition
    5. Bi-substrate reactions
    """
    
    def __init__(self, S, G, met_idx):
        self.S = S
        self.G = G
        self.met_idx = met_idx
        self.n_met, self.n_rxn = S.shape
        self.n_gene = G.shape[0]
        
        # Ground truth kinetic parameters (COMPLEX)
        np.random.seed(456)  # Different seed
        self.true_vmax = np.exp(np.random.randn(self.n_rxn) * 0.5) + 0.5
        self.true_km = np.exp(np.random.randn(self.n_rxn) * 0.5) + 0.3
        
        # Hill coefficients: cooperativity (n > 1 = positive, n < 1 = negative)
        self.hill_n = 1.0 + np.random.rand(self.n_rxn) * 1.5  # Range 1.0-2.5
        
        # Product inhibition constants
        self.ki_product = np.exp(np.random.randn(self.n_rxn) * 0.3) + 1.0
        
        # Substrate inhibition constants (only for some reactions)
        self.ksi = np.exp(np.random.randn(self.n_rxn) * 0.5) + 5.0
        self.has_substrate_inhibition = np.random.rand(self.n_rxn) < 0.3
        
        # Allosteric regulation strength
        self.allosteric_strength = 0.3
        
    def compute_flux(self, M, G_expr):
        """Compute ground truth flux with COMPLEX kinetics."""
        v = np.zeros(self.n_rxn)
        
        # Get key metabolite indices
        atp_idx = self.met_idx.get('ATP', 0)
        adp_idx = self.met_idx.get('ADP', 1)
        nadh_idx = self.met_idx.get('NADH', 4)
        pyruvate_idx = self.met_idx.get('Pyruvate', 9)
        
        # Energy charge: (ATP + 0.5*ADP) / (ATP + ADP + AMP)
        # Simplified: ATP / (ATP + ADP)
        energy_charge = M[atp_idx] / (M[atp_idx] + M[adp_idx] + 0.1)
        
        for j in range(self.n_rxn):
            # Get substrate
            subs = np.where(self.S[:, j] < 0)[0]
            prods = np.where(self.S[:, j] > 0)[0]
            
            if len(subs) > 0:
                S_conc = M[subs[0]]
            else:
                S_conc = 1.0
                
            if len(prods) > 0:
                P_conc = M[prods[0]]
            else:
                P_conc = 0.1
            
            # =====================
            # COMPLEX KINETICS
            # =====================
            
            Vmax = self.true_vmax[j]
            Km = self.true_km[j]
            n = self.hill_n[j]
            Ki = self.ki_product[j]
            
            # 1. Hill kinetics with product inhibition
            # v = Vmax * S^n / (Km^n * (1 + P/Ki) + S^n)
            S_n = S_conc ** n
            Km_n = Km ** n
            v_base = Vmax * S_n / (Km_n * (1 + P_conc / Ki) + S_n + 1e-6)
            
            # 2. Substrate inhibition (for some reactions)
            if self.has_substrate_inhibition[j]:
                Ksi = self.ksi[j]
                # v = v * 1 / (1 + S/Ksi)
                v_base = v_base / (1 + S_conc / Ksi)
            
            # 3. Allosteric regulation based on energy charge
            # Glycolysis (rxns 0-9): inhibited by high energy charge
            # TCA (rxns 10-17): activated by high energy charge
            # Biosynthesis (rxns 18+): activated by high energy charge
            if j < 10:  # Glycolysis
                allosteric_mod = 1 - self.allosteric_strength * (energy_charge - 0.5)
            elif j < 18:  # TCA
                allosteric_mod = 1 + self.allosteric_strength * (energy_charge - 0.5)
            else:  # Biosynthesis
                allosteric_mod = 1 + 0.5 * self.allosteric_strength * (energy_charge - 0.5)
            
            allosteric_mod = np.clip(allosteric_mod, 0.3, 2.0)
            v_base = v_base * allosteric_mod
            
            # 4. NADH feedback (complex coupling)
            # High NADH inhibits oxidative reactions
            nadh_effect = 1 / (1 + (M[nadh_idx] / 2.0) ** 2)
            if 10 <= j < 18:  # TCA reactions
                v_base = v_base * (0.5 + 0.5 * nadh_effect)
            
            v[j] = v_base
        
        # Gene modulation (same as before)
        enzyme_level = 1 / (1 + np.exp(-G_expr @ self.G))
        v = v * enzyme_level
        
        return v
    
    def generate_trajectory(self, n_steps, batch_size, dt=0.1, condition='growth'):
        """Generate ground truth trajectory with complex kinetics."""
        # Initial conditions
        np.random.seed(None)  # Random for each call
        M = np.exp(np.random.randn(batch_size, self.n_met) * 0.3) + 1.0
        G = np.ones((batch_size, self.n_gene)) + np.random.randn(batch_size, self.n_gene) * 0.1
        
        # Condition effects
        if condition == 'stress':
            M = M * 0.5
            # Also perturb gene expression
            G = G * (0.7 + np.random.rand(batch_size, self.n_gene) * 0.6)
        elif condition == 'knockout':
            ko_mask = np.random.rand(batch_size, self.n_gene) < 0.15
            G[ko_mask] = 0.01
        elif condition == 'starvation':
            M = M * 0.2
        elif condition == 'high_energy':
            # High ATP condition
            atp_idx = self.met_idx.get('ATP', 0)
            M[:, atp_idx] = M[:, atp_idx] * 3.0
        
        M_traj = [M.copy()]
        G_traj = [G.copy()]
        
        for _ in range(n_steps - 1):
            # Compute flux for each sample in batch
            v = np.zeros((batch_size, self.n_rxn))
            for i in range(batch_size):
                v[i] = self.compute_flux(M[i], G[i])
            
            # Stoichiometric update
            dM = v @ self.S.T
            M = M + dt * dM + np.random.randn(batch_size, self.n_met) * 0.02
            M = np.clip(M, 0.001, 100.0)
            
            # Gene dynamics with more complex regulation
            dG = -0.01 * (G - 1.0)  # Mean reversion
            # Add metabolite-dependent gene regulation
            atp_idx = self.met_idx.get('ATP', 0)
            energy_signal = np.mean(M[:, atp_idx:atp_idx+2], axis=1, keepdims=True)
            dG = dG + 0.005 * (energy_signal - 2.0)  # Energy-dependent transcription
            
            G = G + dt * dG + np.random.randn(batch_size, self.n_gene) * 0.01
            G = np.clip(G, 0.01, 10.0)
            
            M_traj.append(M.copy())
            G_traj.append(G.copy())
        
        return {
            'metabolites': torch.tensor(np.stack(M_traj, axis=1), dtype=torch.float32),
            'genes': torch.tensor(np.stack(G_traj, axis=1), dtype=torch.float32),
            'condition': condition
        }

# Test complex ground truth
gt_ode = ComplexGroundTruthODE(biochem.S, biochem.G, biochem.met_idx)
test_data = gt_ode.generate_trajectory(30, 4, condition='growth')
print(f"✅ V26.1 FAIR TEST - Complex Ground Truth:")
print(f"   Trajectory shape: {test_data['metabolites'].shape}")
print(f"   Range: [{test_data['metabolites'].min():.2f}, {test_data['metabolites'].max():.2f}]")
print(f"")
print(f"   Complex kinetics features:")
print(f"   - Hill coefficients: {gt_ode.hill_n.min():.2f} to {gt_ode.hill_n.max():.2f}")
print(f"   - Substrate inhibition: {gt_ode.has_substrate_inhibition.sum()}/{gt_ode.n_rxn} reactions")
print(f"   - Allosteric strength: {gt_ode.allosteric_strength}")

In [ ]:
#@title 7️⃣ Configuration

# Training
N_EPOCHS = 1000
BATCH_SIZE = 32
SEQ_LEN = 25
LEARNING_RATE = 3e-3  # Higher LR OK for small model
WEIGHT_DECAY = 1e-4

# Validation
VAL_EVERY = 50
PATIENCE = 15

# Scheduled sampling
TF_START = 0.8
TF_END = 0.0
TF_DECAY_EPOCHS = 400

print("📋 V26 Configuration")

In [ ]:
#@title 8️⃣ Initialize Model

model = DarkManifoldV26(
    S=biochem.S,
    G=biochem.G,
    met_idx=biochem.met_idx,
    hidden_dim=64
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-5
)

gt_ode = ComplexGroundTruthODE(biochem.S, biochem.G, biochem.met_idx)

print(f"✅ Model ready on {device}")

In [ ]:
#@title 9️⃣ Training with Physics Loss

def get_tf_ratio(epoch):
    if epoch >= TF_DECAY_EPOCHS:
        return TF_END
    return TF_START - (TF_START - TF_END) * epoch / TF_DECAY_EPOCHS

def evaluate(model, gt_ode, conditions=['growth', 'stress', 'knockout', 'starvation']):
    model.eval()
    results = {}
    
    with torch.no_grad():
        for cond in conditions:
            data = gt_ode.generate_trajectory(SEQ_LEN, 8, condition=cond)
            M = data['metabolites'][:, 0].to(device)
            G = data['genes'][:, 0].to(device)
            target = data['metabolites'].to(device)
            
            # Autoregressive rollout
            preds = [M]
            for t in range(SEQ_LEN - 1):
                M, G = model(M, G)
                preds.append(M)
            preds = torch.stack(preds, dim=1)
            
            pred_flat = preds.flatten().cpu().numpy()
            target_flat = target.flatten().cpu().numpy()
            corr = np.corrcoef(pred_flat, target_flat)[0, 1]
            results[cond] = corr
    
    model.train()
    return results

# Training
train_losses = []
val_corrs = []
best_val = -1
best_state = None
patience_ctr = 0

print("🚀 Starting V26 training...")

for epoch in tqdm(range(N_EPOCHS)):
    model.train()
    tf = get_tf_ratio(epoch)
    
    # Sample condition
    cond = np.random.choice(['growth', 'stress', 'knockout', 'starvation'])
    data = gt_ode.generate_trajectory(SEQ_LEN, BATCH_SIZE, condition=cond)
    
    M_seq = data['metabolites'].to(device)
    G_seq = data['genes'].to(device)
    
    M = M_seq[:, 0]
    G = G_seq[:, 0]
    
    loss = 0
    for t in range(SEQ_LEN - 1):
        M_pred, G_pred, v_class, v_resid = model(M, G, return_components=True)
        
        M_target = M_seq[:, t + 1]
        G_target = G_seq[:, t + 1]
        
        # Main loss: prediction accuracy
        loss += F.mse_loss(M_pred, M_target)
        loss += 0.5 * F.mse_loss(G_pred, G_target)
        
        # Physics regularization: residual should be SMALL
        residual_penalty = 0.01 * (v_resid ** 2).mean()
        loss += residual_penalty
        
        # Scheduled sampling
        if torch.rand(1).item() < tf:
            M, G = M_target, G_target
        else:
            M, G = M_pred.detach(), G_pred.detach()
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    train_losses.append(loss.item() / SEQ_LEN)
    
    # Validation
    if epoch % VAL_EVERY == 0:
        val_results = evaluate(model, gt_ode)
        avg = np.mean(list(val_results.values()))
        val_corrs.append((epoch, avg, val_results))
        
        print(f"\nEpoch {epoch}: Loss={train_losses[-1]:.4f}, TF={tf:.2f}")
        print(f"   growth={val_results['growth']:.3f}, stress={val_results['stress']:.3f}, "
              f"ko={val_results['knockout']:.3f}, starv={val_results['starvation']:.3f}")
        print(f"   Avg: {avg:.3f}")
        
        if avg > best_val:
            best_val = avg
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
            print(f"   ✨ New best!")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"\n⏹️ Early stopping at epoch {epoch}")
                break

if best_state:
    model.load_state_dict(best_state)
print(f"\n✅ Best validation: {best_val:.3f}")

In [ ]:
#@title 🔟 Final Analysis: Physics vs Learned

model.eval()

# Final evaluation
print("="*60)
print("V26 FINAL RESULTS")
print("="*60)

final_results = {}
for cond in ['growth', 'stress', 'knockout', 'starvation']:
    corrs = []
    for _ in range(5):
        data = gt_ode.generate_trajectory(30, 16, condition=cond)
        with torch.no_grad():
            M = data['metabolites'][:, 0].to(device)
            G = data['genes'][:, 0].to(device)
            
            preds = [M]
            for t in range(29):
                M, G = model(M, G)
                preds.append(M)
            preds = torch.stack(preds, dim=1)
            
            pred_flat = preds.flatten().cpu().numpy()
            target_flat = data['metabolites'].flatten().numpy()
            corr = np.corrcoef(pred_flat, target_flat)[0, 1]
            corrs.append(corr)
    
    final_results[cond] = {'mean': np.mean(corrs), 'std': np.std(corrs)}
    print(f"{cond:12s}: {np.mean(corrs):.3f} ± {np.std(corrs):.3f}")

avg = np.mean([r['mean'] for r in final_results.values()])
print(f"\n{'AVERAGE':12s}: {avg:.3f}")
print(f"\nV24 comparison: {avg:.3f} vs 0.47")
print(f"Improvement: {(avg - 0.47) / 0.47 * 100:+.1f}%")

# Analyze contribution of physics vs learned
print("\n" + "="*60)
print("PHYSICS vs LEARNED CONTRIBUTION")
print("="*60)

with torch.no_grad():
    data = gt_ode.generate_trajectory(10, 4, condition='growth')
    M = data['metabolites'][:, 0].to(device)
    G = data['genes'][:, 0].to(device)
    
    _, _, v_classical, v_residual = model(M, G, return_components=True)
    
    class_mag = v_classical.abs().mean().item()
    resid_mag = v_residual.abs().mean().item()
    total_mag = class_mag + resid_mag
    
    print(f"Classical flux magnitude: {class_mag:.4f} ({100*class_mag/total_mag:.1f}%)")
    print(f"Residual flux magnitude:  {resid_mag:.4f} ({100*resid_mag/total_mag:.1f}%)")
    print(f"\n→ Physics provides {100*class_mag/total_mag:.0f}% of prediction")
    print(f"→ Neural net provides {100*resid_mag/total_mag:.0f}% correction")

In [ ]:
#@title 1️⃣1️⃣ Visualization

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Loss
ax = axes[0, 0]
ax.plot(train_losses, 'b-', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 2. Validation
ax = axes[0, 1]
epochs = [v[0] for v in val_corrs]
corrs = [v[1] for v in val_corrs]
ax.plot(epochs, corrs, 'g-o', markersize=4)
ax.axhline(0.7, color='r', linestyle='--', label='Target: 0.7')
ax.axhline(0.47, color='orange', linestyle='--', label='V24: 0.47')
ax.set_xlabel('Epoch')
ax.set_ylabel('Avg Validation Correlation')
ax.set_title('Validation Performance')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Per-condition
ax = axes[0, 2]
conds = list(final_results.keys())
means = [final_results[c]['mean'] for c in conds]
stds = [final_results[c]['std'] for c in conds]
colors = ['green', 'orange', 'red', 'purple']
ax.bar(conds, means, yerr=stds, color=colors, alpha=0.7, capsize=5)
ax.axhline(0.7, color='r', linestyle='--')
ax.axhline(0.47, color='gray', linestyle='--', label='V24')
ax.set_ylabel('Correlation')
ax.set_title('Multi-Condition Results')
ax.set_ylim(0, 1)

# 4. Trajectory
ax = axes[1, 0]
with torch.no_grad():
    data = gt_ode.generate_trajectory(30, 1, condition='growth')
    M = data['metabolites'][0, 0:1].to(device)
    G = data['genes'][0, 0:1].to(device)
    
    preds = [M.cpu()]
    for t in range(29):
        M, G = model(M, G)
        preds.append(M.cpu())
    preds = torch.stack(preds, dim=1)[0].numpy()
    targets = data['metabolites'][0].numpy()

for i in range(5):
    ax.plot(preds[:, i], '--', alpha=0.7)
    ax.plot(targets[:, i], '-', alpha=0.5)
ax.set_xlabel('Time step')
ax.set_ylabel('Concentration')
ax.set_title('Sample Trajectory')

# 5. Scatter
ax = axes[1, 1]
ax.scatter(targets.flatten(), preds.flatten(), alpha=0.5, s=5)
ax.plot([0, targets.max()], [0, targets.max()], 'r--')
corr = np.corrcoef(targets.flatten(), preds.flatten())[0, 1]
ax.set_xlabel('Target')
ax.set_ylabel('Predicted')
ax.set_title(f'Pred vs Target (r={corr:.3f})')

# 6. Architecture comparison
ax = axes[1, 2]
versions = ['V24', 'V25', 'V26']
params = [8_000_000, 50_000, sum(p.numel() for p in model.parameters())]
eval_corr = [0.47, 0.0, avg]  # V25 untested
ax.bar(versions, params, color=['red', 'orange', 'green'], alpha=0.7)
ax.set_ylabel('Parameters')
ax.set_title('Model Complexity')
ax.set_yscale('log')
for i, (v, p, c) in enumerate(zip(versions, params, eval_corr)):
    ax.annotate(f'r={c:.2f}', (i, p), ha='center', va='bottom')

plt.tight_layout()
plt.savefig('v26_results.png', dpi=150)
plt.show()

In [ ]:
#@title 1️⃣2️⃣ Save Model

torch.save({
    'model_state_dict': model.state_dict(),
    'final_results': final_results,
    'train_losses': train_losses,
    'val_corrs': val_corrs,
    'best_val_corr': best_val,
    'n_params': sum(p.numel() for p in model.parameters()),
    'v26_design': {
        'physics_first': True,
        'neural_residual': True,
        'structured_biochemistry': True,
        'ground_truth_ode': True,
        'scheduled_sampling': True,
        'early_stopping': True
    }
}, 'dark_manifold_v26.pt')

print("✅ Saved to dark_manifold_v26.pt")

# 📌 V26 Summary

## Key Innovation: Physics-First Neural Residual

```
V24 approach:  Prediction = Giant Neural Network
V26 approach:  Prediction = Physics + Small Neural Correction
```

## Architecture Comparison

| Aspect | V24 | V26 |
|--------|-----|-----|
| Parameters | ~8,000,000 | ~10,000 |
| Physics | Learned S@v blend | Hard constraint: dM/dt = S@v |
| Kinetics | Learned from scratch | Michaelis-Menten + residual |
| Neural role | Everything | Small correction only |

## Why This Works

1. **V12 lesson**: Structural priors beat complexity
2. **Physics**: 90% of dynamics is known (stoichiometry, kinetics)
3. **Residual**: Neural net only learns the unknown 10%
4. **Sample efficiency**: Less to learn = better generalization

## Key Design Decisions

1. **Structured biochemistry**: Real pathway organization
2. **Ground truth ODE**: Data has known underlying physics
3. **Residual penalty**: Forces neural net to stay small
4. **Scheduled sampling**: Smooth train→eval transition
5. **Early stopping**: Prevent overfitting